# பாடம் 03 - முகமை வடிவமைப்பு வடிவமைப்புகள்

இந்த பாடத்தில், பயனுள்ள AI முகவர்களை உருவாக்க மூன்று அடிப்படையான வடிவமைப்பு முறைமைகள் பற்றி ஆராய்ச்சிகள் செய்வோம்:

1. **தளரான முகவர் அறிவுரைகள்** — முகவர் நடத்தை வழிநடத்தும் துல்லியமான, பங்கு வரையறுக்கும் பிராம்ப்ட்களை உருவாக்குதல்  
2. **Pydantic மாதிரிகளுடன் கட்டமைக்கப்பட்ட வெளியீடு** — முகவர்கள் கணிக்கப்பட்ட, செல்லுபடியான தரவைத் திரும்ப வழங்குவதை உறுதிசெய்தல்  
3. **ஒரே பொறுப்புத்தன்மை கொண்ட முகவர்கள்** — ஒரு காரியம் நன்கு செய்யும் கவனம் திருப்பப்பட்ட முகவர்கள் வடிவமைத்தல்  

நாம் ஒவ்வொரு வடிவமைப்பையும் ஒரு **பயண இட பரிந்துரையாளர்** காட்சியில் பயன்படுத்தி, இடங்கள் பரிந்துரை செய்தல், கிடைப்பை சோதனை செய்தல் மற்றும் ரசாயன அமைப்புகளை கையாளும் ஒரு அமைப்பை திருநிலைபடுத்தி உருவாக்குவோம்.


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## மாதிரி 1: தெளிவான முகவர் வழிமுறைகள்

அதிக பாதிப்புள்ள மாதிரி மிகவும் எளிமையானது: உங்கள் முகவருக்கான தெளிவான, விரிவான வழிமுறைகளை எழுதுதல்.

நல்ல வழிமுறைகள் வரையறுக்கின்றன:
- **யார்** அந்த முகவர் (பாத்திரம் மற்றும் மொழி)
- **என்ன** அவர் செய்ய வேண்டும் (படி படியான பொறுப்புக்கள்)
- **எப்படி** அவர் நடப்பார் (கட்டுப்பாடுகள் மற்றும் நடைமுறை)

கீழே, நாம் ஒரு பயணக் கான்சியர்ஜ் முகவரைக் குறித்து தெளிவான வழிமுறைகளுடன் உருவாக்குகிறோம், அவை அவன் உருவாக்கும் ஒவ்வொரு பதிலும் வடிவமைக்கின்றன.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic மாதிரிகளுடன் கட்டமைக்கப்பட்ட வெளியீடு

வார்த்தை உரை பேச்சுவார்த்தைக்குப் பயன்படுகிறது, ஆனால் கீழ்நிலை அமைப்புகள் கட்டமைக்கப்பட்ட தரவை தேவைப்படுகின்றன.  
**Pydantic மாதிரிகள்** மற்றும் **கருவி செயல்பாடு** ஆகியவற்றை இணைத்துச் செயல்படுத்துவதன் மூலம், நாம்:

- முகவரியின் வெளியீட்டிற்கான துல்லியமான திட்டவட்டத்தை வரையறுக்கலாம்  
- பதில்களை தானாக சரிபார்க்கலாம்  
- முகவரியின் முடிவுகளை விண்ணப்ப தர்க்கச்சார்ந்த முறையில் ஒருங்கிணைக்கலாம்  

உண்மையான தரவுகளின் அடிப்படையில் முகவர் பரிந்துரைகளை வலுப்படுத்த ஒரு இலக்க bestemming விவரங்களை வழங்கும் ஒரு கருவியை வழங்குகிறோம்.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: ஒரே பொறுப்பு முகவர்கள்

சிக்கலான பணிகள் பல கவனம் செலுத்தும் முகவர்கள் மத்தியில் வேலையைப் பிரிக்கும் மூலம் நன்மை பெருக்குகின்றன, ஒவ்வொன்றும் ஒரு தனிப்பட்ட பொறுப்பை கொண்டுள்ளனர்:

- இடங்கள் மற்றும் கிடைக்கும் வாய்ப்புகளை தெரிந்துகொள்ளும் **இடம் நிபுணர்**
- விமானங்கள், விடுதிகள் மற்றும் பயணத் திட்டங்களை நிர்வகிக்கும் **வணிகத் திட்டமிடுபவர்**

இது *கவன வித்தியாசம்* என்ற மென்பொருள் பொறியியல் முதன்மையை பிரதிபலிக்கிறது — ஒவ்வொரு முகவர் தனித்து சோதனை செய்தல், பராமரித்தல், மற்றும் மேம்படுத்தல் எளிதாகிறது.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## சுருக்கம்

இந்த பாடத்தில் நாம் ஒரு பயண பரிந்துரையாளர் சூழலுக்கு மூன்று முகவர் வடிவமைப்பு வடிவங்களை பயன்படுத்தினோம்:

| வடிவம் | முக்கிய கருத்து | பயன் |
|---|---|---|
| **தெளிவான அறிவுறுத்தல்கள்** | முன்னதாகவே தனிப்பட்ட நபர், பொறுப்புகள் மற்றும் கட்டுப்பாடுகளை விவரிக்கவும் | ஒருமுறையான, தயாரிப்புக்கு ஏற்ப முகவர் நடத்தை |
| **கட்டமைக்கப்பட்ட வெளியீடு** | பதிலான வடிவமாக Pydantic மாதிரிகளை பயன்படுத்தவும் | சரிபார்க்கப்பட்ட, இயந்திரம் படிக்கக்கூடிய முடிவுகள் |
| **ஒரே பொறுப்பு** | ஒவ்வொரு முகவருக்கும் ஒரு கவனம்வாய்ந்த வேலை வழங்கவும் | சுலபமாக சோதிக்கவும், பராமரிக்கவும், மற்றும் சேர்க்கவும் |

இந்த வடிவங்கள் இயல்பாக சேர்க்கப்படுகின்றன — நீங்கள் தெளிவான அறிவுறுத்தல்களை கட்டமைக்கப்பட்ட வெளியீட்டுடன் ஒரு ஒரே-பொறுப்பு முகவருக்குள் இணைத்து வலுவான, தயாரிப்பு-தயார் அமைப்புகளை உருவாக்கலாம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
